(course-alzheimer-final-project)=

# Path A - Module 54: Final Project: From Sequence to Analysis

This capstone brings the course tools together around the Amyloid-Beta fibril
and the candidate peptide **KLVFF**. Keep the distinction between constructing a
candidate configuration and demonstrating a biological effect.

## Your Final Mission
Prepare the two partners, supply a reviewed candidate pose, then characterize
the assembled geometry. The preparation statements have bounded local controls;
the interface recipe in Step 5 has independent analytic regression controls.
These controls do not execute or qualify the complete 2BEG download, repair,
pose selection, parameterization, solvation and relaxation route.
This remains a workflow outline, rather than a ready-to-run biological
preparation protocol. In particular, Step 3 requires you to supply coordinates
for the final prepared candidate before it can continue. Solvation does not dock
a molecule or establish a binding pose.


## Step 1: Target Preparation

Load source structure index 0 and explicitly choose native connectivity
inference. Retain each independent repair result: hydrogen preparation must
consume the heavy-atom repair output, and all later steps must use its result.
The native heavy-atom repair leaves unsupported gaps unassessed with a diagnostic.
OpenMM's legacy pH route supplies hydrogen inventory/placement under its own
rules; pH 7.4 is an explicit preparation choice, not an environmental protonation
prediction or a fixed-state guarantee.

Inspect the returned heavy-template coverage and unresolved chemistry before
proceeding. A successful call does not certify clashes, hydrogen directions,
terminal chemistry or a simulation-ready target. Repair may change atom indices;
record partner populations only after these operations. See
{ref}`Tutorial_Residue_Chemical_Coverage` and
{ref}`Tutorial_Add_missing_hydrogens`.


In [ ]:
import molsysmt as msm

# 1. Load and Audit
fibril = msm.convert(
    'pdb:2BEG', to_form='molsysmt.MolSys', structure_indices=0,
    get_missing_bonds=True, bond_inference_engine='MolSysMT'
)
fibril = msm.build.add_missing_heavy_atoms(fibril, engine='MolSysMT')
fibril = msm.build.add_missing_hydrogens(fibril, pH=7.4, engine='OpenMM')
target_coverage = msm.build.get_residue_chemical_coverage(
    fibril, structure_indices=0
)
print("Target heavy-template coverage:", target_coverage['summary'])


## Step 2: Candidate Construction

Build KLVFF, then use the separate terminal tool to add the explicitly chosen
ACE and NME groups. Retain its independent output and the subsequent hydrogen
preparation output. The builder does not accept terminal/conformation keywords.

Use the general chemical-state quartet and structural rotation tools to request
illustrative extended backbone targets: **phi = -135 degrees** and
**psi = 135 degrees**. These are declared geometric inputs, not a prediction of
a beta sheet, an energy minimum or a bound conformation. Side-chain rotamers,
steric quality, chemistry and environmental hydrogen geometry need independent
review. Capping and preparation precede quartet lookup so the rotations use the
current atom axis, including the caps and observed hydrogens. See
{ref}`Tutorial_Add_missing_terminal_cappings` and
{ref}`Tutorial_Set_dihedral_angles`.

Before Step 3, replace `candidate_pose_coordinates = None` with a length quantity
for **one reviewed pose of this final prepared peptide**, in its current atom
order. You can provide shape `(n_candidate_atoms, 3)` or
`(1, n_candidate_atoms, 3)` through the public coordinate setter. Preserve the
map from the tool or source that produced that pose. The example does not obtain
this pose by merging molecules or by solvating them.


In [ ]:
# 2. Build the peptide
peptide = msm.build.build_peptide('KLVFF', engine='MolSysMT')
peptide = msm.build.add_missing_terminal_cappings(
    peptide, N_terminal='ACE', C_terminal='NME', engine='MolSysMT'
)
peptide = msm.build.add_missing_hydrogens(peptide, pH=7.4, engine='OpenMM')
phi_quartets = msm.chemical_states.get_dihedral_quartets(peptide, phi=True)
psi_quartets = msm.chemical_states.get_dihedral_quartets(peptide, psi=True)
peptide = msm.structure.set_dihedral_angles(
    peptide, dihedral_quartets=phi_quartets, angles='-135 degrees', pbc=False
)
peptide = msm.structure.set_dihedral_angles(
    peptide, dihedral_quartets=psi_quartets, angles='135 degrees', pbc=False
)
candidate_pose_coordinates = None  # Supply a reviewed final-candidate pose before Step 3.
print("Candidate constructed with the declared caps and illustrative torsion targets.")


## Step 3: Systems Assembly & Solvation

Supply the prepared candidate's reviewed coordinates in the target's Cartesian
coordinate system. This step copies the candidate and assigns that quantity
through `msm.set`; it preserves the independently prepared candidate. There is
no docking or automatic pose search. Both partners must be whole and in the
intended common image, with their chemistry and steric quality reviewed.
The explicit missing-pose check prevents this outline from silently using an
unplaced builder configuration.

Join the dry partners and record explicit atom-index populations in merge order.
The target and candidate can both be classified as peptides, so molecular type
is not their identity. The example binds an **AMBER14/TIP3P** mechanics request
before solvation; this declares the requested model rather than certifying that
the biological partners match its templates. OpenMM must accept the complete
prepared topology. The bounded control receives this model for the synthetic
candidate, not for the full 2BEG complex. Review all other mechanics settings and
the actual backend admission before any full-project simulation.

Ordinary OpenMM solvation retains source atoms as the leading output prefix.
The ranges below therefore survive that operation and the following coordinate-only
minimization. Extraction, reordering, atom removal or another preparation route
needs its own correspondence map. Keep the dry-partner maps with the result;
do not use pre-repair or pre-capping indices. Adding water and salt does not
equilibrate this model. The complete biological solvation branch remains
unexecuted here.


In [ ]:
# 3. Merge and Solvate
if candidate_pose_coordinates is None:
    raise ValueError("Supply candidate_pose_coordinates for the final prepared peptide before assembly.")
peptide_placed = msm.copy(peptide)
msm.set(peptide_placed, structure_indices=0, coordinates=candidate_pose_coordinates)
n_target_atoms = msm.get(fibril, n_atoms=True)
n_candidate_atoms = msm.get(peptide_placed, n_atoms=True)
target_atom_indices = list(range(n_target_atoms))
candidate_atom_indices = list(range(n_target_atoms, n_target_atoms + n_candidate_atoms))
complex = msm.merge([fibril, peptide_placed])
complex = msm.convert(
    [complex, {'forcefield': 'AMBER14', 'water_model': 'TIP3P'}],
    to_form='molsysmt.MolSys'
)
final_complex = msm.build.solvate(
    complex, engine='OpenMM', box_shape='cubic', clearance='1.0 nm',
    ionic_strength='150 mM'
)
print(f"Complex assembled and solvated with {msm.get(final_complex, n_atoms=True)} atoms.")


## Step 4: Physics Relaxation
Use a parameterized and spatially assembled complex. Retain the independently minimized result for later analysis. Local optimization alone does not ensure that clashes are absent or establish binding stability.

The independent native workflow below differs from a live engine object: unsupported engine-copy requests are rejected before optimization rather than changing the input before failing.


In [ ]:
# 4. Energy Minimization
final_complex = msm.molecular_mechanics.potential_energy_minimization(final_complex)
print("Energy relaxation completed.")

## Step 5: Geometric Interface Characterization

Analyze structure index 0 with the target and candidate populations declared
above. First extract their union, excluding solvent and other background atoms.
Then extract each partner independently: `get_sasa(selection=...)` only filters
outputs and would keep the other atoms as occluders.

At the same coordinates, with the same native van der Waals radii, a **0.14 nm
probe radius** and **960 sphere points per atom**, calculate
$\Delta SASA = SASA(A) + SASA(B) - SASA(A\cup B)$.
Report both the **total SASA loss** and its half, the **mean interface area**.
The half-area convention follows the [PDBePISA interface-area definition](https://www.ebi.ac.uk/pdbe/prot_int/pi_tips.html).
This composition uses MolSysMT's existing Shrake–Rupley calculation
([Shrake and Rupley, 1973](https://doi.org/10.1016/0022-2836(73)90011-9)); it
does not execute PISA or reproduce its radii, assembly analysis or energetics.

The recipe explicitly uses **Cartesian geometry** (`pbc=False`) for SASA and
hydrogen-bond screening. Both partners must be whole and in the intended common
image; it does not reconstruct a periodic interface. Exclude explicit solvent
from this two-partner area definition. Isolated-partner calculations retain
the bound coordinates and perform no separate relaxation. Results depend on
radii, probe, sampled structure and quadrature; study sampling convergence for
quantitative applications.

`get_area_buried` is a Rose reference-table lookup and `get_buried_fraction`
is a Janin reference-table lookup. Neither calculates this geometric interface.
The hydrogen-bond count below uses the explicit-H distance criterion with a
0.23 nm H–acceptor cutoff. Its interpretation requires the supplied hydrogen
inventory. Neither that count nor SASA loss proves binding stability, affinity,
free energy, equilibration or a favorable biological effect. See
{ref}`Tutorial_Get_sasa` and {ref}`Tutorial_Get_buch_hbonds`.


In [ ]:
# 5. Geometric SASA loss and observed H-bond count
if len(target_atom_indices) == 0 or len(candidate_atom_indices) == 0:
    raise ValueError("Both interface populations must be nonempty.")
partner_atom_indices = list(target_atom_indices) + list(candidate_atom_indices)
if len(set(partner_atom_indices)) != len(partner_atom_indices):
    raise ValueError("Interface populations must be unique and disjoint.")

# Extraction preserves this explicit order and gives the dry pair its local indices.
molsys_AB = msm.extract(final_complex, selection=partner_atom_indices, structure_indices=0)
local_A = list(range(len(target_atom_indices)))
local_B = list(range(len(target_atom_indices), len(partner_atom_indices)))
sasa_options = dict(
    engine='MolSysMT', element='atom', probe_radius='0.14 nm',
    n_sphere_points=960, pbc=False, use_gpu=False
)
sasa_A = msm.physchem.get_sasa(msm.extract(molsys_AB, selection=local_A), **sasa_options).sum(axis=1)
sasa_B = msm.physchem.get_sasa(msm.extract(molsys_AB, selection=local_B), **sasa_options).sum(axis=1)
sasa_AB = msm.physchem.get_sasa(molsys_AB, **sasa_options).sum(axis=1)
delta_sasa = sasa_A + sasa_B - sasa_AB
interface_mean_area = delta_sasa / 2

hbonds, _ = msm.interactions.hbonds.get_buch_hbonds(
    molsys_AB, selection=local_A, selection_2=local_B,
    structure_indices=0, distance_threshold='0.23 nm', pbc=False
)
n_hbonds = len(hbonds[0])
puw = msm.pyunitwizard
print("CONFIGURATION REPORT (structure index 0):")
print(f" - Total SASA loss (nm²): {puw.get_value(delta_sasa, to_unit='nm**2')[0]:.6f}")
print(f" - Mean interface area (nm²): {puw.get_value(interface_mean_area, to_unit='nm**2')[0]:.6f}")
print(f" - H-bonds matching the distance criterion: {n_hbonds}")


## Final Words

You have combined preparation, coordinate editing and explicit geometric
observables in the **Path of the Alzheimer's Architect**. The bounded controls
check public preparation calls and result ownership, declared candidate torsions,
pose assignment/merge maps and the interface's surface definition, units and
selections. They do not qualify the full biological preparation or demonstrate
inhibition. Keep the exact prepared model, pose correspondence, partner maps,
methods and parameters with any reported observations. Qualify the complete
workflow independently before treating it as ready for biological use.
